# 01 — MSP-Podcast 4クラス特徴cacheの作成

MSP-Podcast R1.10の監査、strict manifest作成、実音声1件のCPU benchmark、容量+20%判定、emotion2vec特徴抽出、cache検証を上から順に行います。

実データを読む処理はすべて既定で無効です。設定セルでパスを指定し、各段階の結果を確認してから、対応するフラグを1つずつ`True`にしてください。IEMOCAPは今回の一括研究経路には含めません。


In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.cache import validate_cache
from ser_pipeline.exclusions import load_msp_missing_audio_exclusion_contract
from ser_pipeline.features import Emotion2vecEncoder, extract_feature_cache
from ser_pipeline.manifest import (
    audit_dataset, build_manifest, generate_msp_missing_audio_exclusion_contract,
    load_manifest, validate_manifest,
)
from ser_pipeline.notebook_api import environment_summary, mapping_summary, split_summary
from ser_pipeline.preflight import (
    benchmark_audio_extraction, disk_capacity_gate,
    estimate_full_extraction, save_benchmark,
)

STUDY_DATASETS = ('msp_podcast', 'hcudb1')

# 実行フラグ: 最初はすべてFalseのまま、上から1段階ずつ有効化します。
RUN_MSP_AUDIT = False
RUN_MSP_GENERATE_EXCLUSION_CONTRACT = False
RUN_MSP_VERIFY_EXCLUSION_CONTRACT = False
RUN_MSP_BUILD_MANIFEST = False
RUN_MSP_VALIDATE_MANIFEST = False
RUN_MSP_BENCHMARK = False
RUN_MSP_CAPACITY_GATE = False
RUN_FULL_EXTRACTION = False
RUN_VALIDATE_CACHE = False

# 前段の結果をユーザーが確認した後だけTrueにします。
CONFIRM_MANIFEST_VALIDATED = False
CONFIRM_BENCHMARK_AND_CAPACITY = False

# 環境変数を使わない場合は、ここへWSLから見えるPathを直接指定できます。
MSP_ROOT = Path(os.environ['MSP_PODCAST_ROOT']) if os.environ.get('MSP_PODCAST_ROOT') else None
BENCHMARK_AUDIO = Path(os.environ['MSP_BENCHMARK_AUDIO']) if os.environ.get('MSP_BENCHMARK_AUDIO') else None

USER_DIR = PROJECT_ROOT / 'upstream'
CHECKPOINT_PATH = PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'emotion2vec_base.pt'
MANIFEST_PATH = PROJECT_ROOT / 'runs' / 'ser_manifests' / 'msp_podcast_4class_v1.jsonl'
EXCLUSION_CONTRACT_PATH = PROJECT_ROOT / 'runs' / 'ser_manifests' / 'msp_missing_audio_exclusions_v1.json'
CACHE_ROOT = PROJECT_ROOT / 'runs' / 'ser_feature_cache' / 'msp_podcast_base_final_v1'
REPORT_DIR = PROJECT_ROOT / 'runs' / 'ser_feature_preflight' / 'msp_podcast'
BENCHMARK_PATH = REPORT_DIR / 'one_audio_cpu_benchmark.json'
CAPACITY_PATH = REPORT_DIR / 'capacity_estimate.json'


def require_path(value, label, *, kind):
    if value is None:
        raise ValueError(f'Set {label} in the configuration cell before execution')
    path = Path(value)
    valid = path.is_file() if kind == 'file' else path.is_dir()
    if not valid:
        raise FileNotFoundError(f'{label} was not found: {path}')
    return path


def persist_report(report, path):
    save_benchmark(report, path)
    return report


## 1. 設定・実行環境・固定契約


In [ ]:
{
    'msp_root': str(MSP_ROOT) if MSP_ROOT else None,
    'benchmark_audio': str(BENCHMARK_AUDIO) if BENCHMARK_AUDIO else None,
    'user_dir': str(USER_DIR),
    'checkpoint': str(CHECKPOINT_PATH),
    'manifest': str(MANIFEST_PATH),
    'exclusion_contract': str(EXCLUSION_CONTRACT_PATH),
    'cache_root': str(CACHE_ROOT),
    'device': 'cpu',
    'run_flags': {
        'audit': RUN_MSP_AUDIT,
        'generate_exclusion_contract': RUN_MSP_GENERATE_EXCLUSION_CONTRACT,
        'verify_exclusion_contract': RUN_MSP_VERIFY_EXCLUSION_CONTRACT,
        'build_manifest': RUN_MSP_BUILD_MANIFEST,
        'validate_manifest': RUN_MSP_VALIDATE_MANIFEST,
        'benchmark': RUN_MSP_BENCHMARK,
        'capacity_gate': RUN_MSP_CAPACITY_GATE,
        'full_extraction': RUN_FULL_EXTRACTION,
        'validate_cache': RUN_VALIDATE_CACHE,
    },
}


In [ ]:
environment_summary()


In [ ]:
mapping_rows = [row for row in mapping_summary() if row['dataset'] in STUDY_DATASETS]
pd.DataFrame(mapping_rows)


In [ ]:
all_split_contracts = split_summary()
{name: all_split_contracts[name] for name in STUDY_DATASETS}


## 2. MSP-Podcast metadata・音声inventory監査

`RUN_MSP_AUDIT = True`にした場合だけ実データを読みます。添付の利用不能候補リストは除外条件に使用しません。結果には、現在不足している対象音声の元ラベル、4クラス変換後ラベル、公式split別件数も含まれます。`missing_eligible_audio == 874`と固定内訳、`unregistered_audio_files == 0`、現行契約の対象25,985件を確認してから次へ進みます。


In [ ]:
if RUN_MSP_AUDIT:
    msp_root = require_path(MSP_ROOT, 'MSP_ROOT', kind='directory')
    audit_report = audit_dataset('msp_podcast', msp_root)
    persist_report(audit_report, REPORT_DIR / 'audit.json')
else:
    audit_report = {'status': 'disabled_by_default'}
audit_report


### 2.1 現在不足している対象音声の感情・split内訳

添付リストではなく、metadata上の4クラス対象と現在存在するWAVを照合し、不足している対象音声だけを集計します。`missing_count`の合計が`missing_eligible_audio`と一致することを確認してください。


In [ ]:
if RUN_MSP_AUDIT:
    label_pairs = (
        ('A', 'anger'),
        ('H', 'happy'),
        ('S', 'sadness'),
        ('D', 'disgust'),
    )
    missing_label_summary = pd.DataFrame([
        {
            'original_label': original_label,
            'mapped_label': mapped_label,
            'eligible_total': audit_report['eligible_mapped_label_counts'].get(mapped_label, 0),
            'available_count': audit_report['available_eligible_mapped_label_counts'].get(mapped_label, 0),
            'missing_count': audit_report['missing_eligible_original_label_counts'].get(original_label, 0),
        }
        for original_label, mapped_label in label_pairs
    ])
    missing_split_summary = pd.DataFrame([
        {
            'source_split': source_split,
            'missing_count': audit_report['missing_eligible_source_split_counts'].get(source_split, 0),
        }
        for source_split in ('Train', 'Development', 'Test1')
    ])
    print('不足対象音声の感情ラベル内訳:')
    display(missing_label_summary)
    print('不足対象音声の公式split内訳:')
    display(missing_split_summary)
else:
    print('監査は無効です。RUN_MSP_AUDIT = Trueで監査セルから実行してください。')


## 3. 除外候補生成

`RUN_MSP_GENERATE_EXCLUSION_CONTRACT = True`にした場合だけ、添付リストを参照せず、metadata上の4クラス対象と現在のWAV inventoryから不足行を再計算します。874件・固定内訳に一致しない場合はJSONを書きません。


In [ ]:
if RUN_MSP_GENERATE_EXCLUSION_CONTRACT:
    msp_root = require_path(MSP_ROOT, 'MSP_ROOT', kind='directory')
    exclusion_generation_report = generate_msp_missing_audio_exclusion_contract(
        msp_root,
        EXCLUSION_CONTRACT_PATH,
    )
    persist_report(exclusion_generation_report, REPORT_DIR / 'exclusion_contract_generation.json')
else:
    exclusion_generation_report = {'status': 'disabled_by_default'}
exclusion_generation_report


## 4. 除外契約の件数・内訳・SHA確認

`RUN_MSP_VERIFY_EXCLUSION_CONTRACT = True`にすると、874件、元ラベル`A 378 / H 392 / S 80 / D 24`、公式split`Train 520 / Development 210 / Test1 144`、ファイル名順、重複なし、正規化SHA-256を検証します。


In [ ]:
if RUN_MSP_VERIFY_EXCLUSION_CONTRACT:
    contract_path = require_path(EXCLUSION_CONTRACT_PATH, 'EXCLUSION_CONTRACT_PATH', kind='file')
    _, exclusion_verification_report = load_msp_missing_audio_exclusion_contract(contract_path)
    persist_report(exclusion_verification_report, REPORT_DIR / 'exclusion_contract_verification.json')
else:
    exclusion_verification_report = {'status': 'disabled_by_default'}
exclusion_verification_report


## 5. 承認SHA設定

上の検証結果を確認後、承認する`normalized_sha256`を64桁の小文字16進文字列として設定します。未設定のままstrict manifest作成を有効化すると必ず拒否します。


In [ ]:
APPROVED_MSP_EXCLUSION_SHA256 = None
# 例: APPROVED_MSP_EXCLUSION_SHA256 = '64桁の検証済みSHA-256'


## 6. strict manifest作成

`RUN_MSP_BUILD_MANIFEST = True`にすると、承認SHAと除外契約を照合し、契約内874件だけを`included: false`にします。一覧外欠損、復旧済み契約対象、metadata不一致、音声デコード失敗、重複、話者漏洩、ラベル契約違反があれば停止します。


In [ ]:
if RUN_MSP_BUILD_MANIFEST:
    if APPROVED_MSP_EXCLUSION_SHA256 is None:
        raise RuntimeError('Set APPROVED_MSP_EXCLUSION_SHA256 after reviewing the exclusion contract')
    msp_root = require_path(MSP_ROOT, 'MSP_ROOT', kind='directory')
    require_path(EXCLUSION_CONTRACT_PATH, 'EXCLUSION_CONTRACT_PATH', kind='file')
    manifest_build_report = build_manifest(
        'msp_podcast',
        msp_root,
        MANIFEST_PATH,
        strict=True,
        inspect_excluded_audio=True,
        approved_exclusion_contract=EXCLUSION_CONTRACT_PATH,
        expected_exclusion_sha256=APPROVED_MSP_EXCLUSION_SHA256,
    )
    persist_report(manifest_build_report, REPORT_DIR / 'manifest_build.json')
else:
    manifest_build_report = {'status': 'disabled_by_default'}
manifest_build_report


## 7. manifestと実音声の完全検証

`RUN_MSP_VALIDATE_MANIFEST = True`にすると、included音声のmetadataとSHA-256を再計算します。結果が`status: ok`で、`audio.verified_audio`と`included`が一致したことを確認してください。


In [ ]:
if RUN_MSP_VALIDATE_MANIFEST:
    msp_root = require_path(MSP_ROOT, 'MSP_ROOT', kind='directory')
    require_path(MANIFEST_PATH, 'MANIFEST_PATH', kind='file')
    manifest_validation_report = validate_manifest(MANIFEST_PATH, audio_root=msp_root)
    persist_report(manifest_validation_report, REPORT_DIR / 'manifest_validation.json')
else:
    manifest_validation_report = {'status': 'disabled_by_default'}
manifest_validation_report


## 8. 実音声1件のCPU benchmark

manifestで`included: true`のWAVを`BENCHMARK_AUDIO`へ指定します。manifest検証結果を確認後、`CONFIRM_MANIFEST_VALIDATED = True`と`RUN_MSP_BENCHMARK = True`にします。


In [ ]:
if RUN_MSP_BENCHMARK:
    if not CONFIRM_MANIFEST_VALIDATED:
        raise RuntimeError('Confirm the complete MSP manifest validation first')
    benchmark_audio = require_path(BENCHMARK_AUDIO, 'BENCHMARK_AUDIO', kind='file')
    require_path(USER_DIR, 'USER_DIR', kind='directory')
    require_path(CHECKPOINT_PATH, 'CHECKPOINT_PATH', kind='file')
    benchmark_report = benchmark_audio_extraction(
        benchmark_audio,
        USER_DIR,
        CHECKPOINT_PATH,
        device='cpu',
    )
    persist_report(benchmark_report, BENCHMARK_PATH)
else:
    benchmark_report = {'status': 'disabled_by_default'}
benchmark_report


## 9. 全件所要時間・容量+20%ゲート

`RUN_MSP_CAPACITY_GATE = True`にすると、manifestの対象総時間と1件benchmarkから全件見積りを作ります。`capacity.passes`が`True`でなければ全件抽出へ進みません。


In [ ]:
if RUN_MSP_CAPACITY_GATE:
    if not CONFIRM_MANIFEST_VALIDATED:
        raise RuntimeError('Confirm the complete MSP manifest validation first')
    require_path(MANIFEST_PATH, 'MANIFEST_PATH', kind='file')
    require_path(BENCHMARK_PATH, 'BENCHMARK_PATH', kind='file')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    included_rows = [row for row in load_manifest(MANIFEST_PATH) if row['included']]
    total_duration_seconds = sum(float(row['duration_seconds']) for row in included_rows)
    saved_benchmark = json.loads(BENCHMARK_PATH.read_text(encoding='utf-8'))
    estimate = estimate_full_extraction(
        total_duration_seconds,
        saved_benchmark,
        storage_margin=1.2,
    )
    capacity = disk_capacity_gate(CACHE_ROOT, estimate['required_bytes_with_margin'])
    capacity_report = {
        'included_utterances': len(included_rows),
        'estimate': estimate,
        'capacity': capacity,
    }
    persist_report(capacity_report, CAPACITY_PATH)
    if not capacity['passes']:
        raise RuntimeError('Capacity gate failed; do not start full extraction')
else:
    capacity_report = {'status': 'disabled_by_default'}
capacity_report


## 10. MSP-Podcast全件特徴抽出

manifest検証、benchmark、容量判定を確認した後だけ、2つの確認フラグと`RUN_FULL_EXTRACTION`を`True`にします。deviceはCPU、layerは`final`固定です。中断した場合はcacheを削除せず、同じ設定でこのセルを再実行すると完成済みshardを再利用します。


In [ ]:
if RUN_FULL_EXTRACTION:
    if not CONFIRM_MANIFEST_VALIDATED:
        raise RuntimeError('Confirm the complete MSP manifest validation first')
    if not CONFIRM_BENCHMARK_AND_CAPACITY:
        raise RuntimeError('Confirm the CPU benchmark and +20% capacity gate first')
    msp_root = require_path(MSP_ROOT, 'MSP_ROOT', kind='directory')
    require_path(MANIFEST_PATH, 'MANIFEST_PATH', kind='file')
    require_path(USER_DIR, 'USER_DIR', kind='directory')
    require_path(CHECKPOINT_PATH, 'CHECKPOINT_PATH', kind='file')
    saved_capacity = json.loads(require_path(CAPACITY_PATH, 'CAPACITY_PATH', kind='file').read_text(encoding='utf-8'))
    if not saved_capacity['capacity']['passes']:
        raise RuntimeError('Saved capacity gate does not pass')
    saved_benchmark = json.loads(require_path(BENCHMARK_PATH, 'BENCHMARK_PATH', kind='file').read_text(encoding='utf-8'))
    encoder = Emotion2vecEncoder(
        USER_DIR,
        CHECKPOINT_PATH,
        layer='final',
        device='cpu',
    )
    if encoder.info.checkpoint_sha256 != saved_benchmark['encoder_checkpoint_sha256']:
        raise RuntimeError('Benchmark and extraction checkpoint SHA-256 differ')
    extraction_report = extract_feature_cache(
        MANIFEST_PATH,
        msp_root,
        CACHE_ROOT,
        encoder,
        layer='final',
        max_shard_frames=65536,
        expected_dim=768,
    )
    persist_report(extraction_report, REPORT_DIR / 'extraction_result.json')
else:
    extraction_report = {'status': 'disabled_by_default'}
extraction_report


## 11. cache最終検証

抽出完了後に`RUN_VALIDATE_CACHE = True`として実行します。manifest対象件数、768次元、有限float32、shard/index hash、全splitの`_SUCCESS`、cache完了フラグ、checkpoint SHA-256を検証します。


In [ ]:
if RUN_VALIDATE_CACHE:
    require_path(MANIFEST_PATH, 'MANIFEST_PATH', kind='file')
    require_path(CACHE_ROOT, 'CACHE_ROOT', kind='directory')
    saved_benchmark = json.loads(require_path(BENCHMARK_PATH, 'BENCHMARK_PATH', kind='file').read_text(encoding='utf-8'))
    cache_validation = validate_cache(
        CACHE_ROOT,
        MANIFEST_PATH,
        expected_signature={
            'feature_dim': 768,
            'feature_layer': 'final_after_encoder_norm',
            'dtype': 'float32',
        },
    )
    expected_count = sum(bool(row['included']) for row in load_manifest(MANIFEST_PATH))
    if cache_validation['utterances'] != expected_count:
        raise RuntimeError('Cache utterance count does not match the manifest')
    cache_meta = json.loads((CACHE_ROOT / 'cache_meta.json').read_text(encoding='utf-8'))
    if cache_meta['encoder_checkpoint_sha256'] != saved_benchmark['encoder_checkpoint_sha256']:
        raise RuntimeError('Benchmark and cache checkpoint SHA-256 differ')
    partial_files = [str(path) for path in CACHE_ROOT.rglob('*.partial')]
    if partial_files:
        raise RuntimeError(f'Partial cache files remain: {partial_files[:5]}')
    cache_validation_report = {
        **cache_validation,
        'checkpoint_sha256': cache_meta['encoder_checkpoint_sha256'],
        'partial_files': partial_files,
    }
    persist_report(cache_validation_report, REPORT_DIR / 'cache_validation.json')
else:
    cache_validation_report = {'status': 'disabled_by_default'}
cache_validation_report
